In [1]:
import pandas as pd

For questions 3 to 5:  
Given a dataset with treatment and control data having “before” and “after” parts, apply a differences-in-differences regression.  
Use homework_3.2.a.csv and homework_3.2.b.csv. 

In [2]:
df_a = pd.read_csv('homework_3.2.a.csv')
df_a

,group1,time1,outcome1
0,0,0,0.882026
1,0,1,1.600079
2,0,0,0.489369
3,0,1,2.520447
4,0,0,0.933779
...,...,...,...
995,1,1,4.306435
996,1,0,1.900801
997,1,1,4.147096
998,1,0,1.426195


In [3]:
df_b = pd.read_csv('homework_3.2.b.csv')
df_b

,group2,time2,outcome2
0,0,0,0.667155
1,0,1,2.470969
2,0,0,-0.506778
3,0,1,1.525657
4,0,0,0.273664
...,...,...,...
995,1,1,4.637301
996,1,0,3.681828
997,1,1,4.710121
998,1,0,0.629718


In [ ]:
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [9]:
# Dataset A
df_a['interaction1'] = df_a['group1'] * df_a['time1']

model_a = smf.ols(
    "outcome1 ~ group1 + time1 + interaction1",
    data=df_a
).fit(cov_type="HC1")

print(model_a.summary())

# differences in differences
mean_a = df_a.groupby(['group1', 'time1'])['outcome1'].mean()
did_a = (mean_a[1][1] - mean_a[1][0]) - (mean_a[0][1] - mean_a[0][0])

print("Differences in Differences for Dataset A")
print(f"Control (0) Before (0): {mean_a[0][0]:.4f}")
print(f"Control (0) After (1):  {mean_a[0][1]:.4f}")
print(f"Treated (1) Before (0): {mean_a[1][0]:.4f}")
print(f"Treated (1) After (1):  {mean_a[1][1]:.4f}")
print(f"Treatment Effect: {did_a:.4f}")


                            OLS Regression Results                            
Dep. Variable:               outcome1   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.899
Method:                 Least Squares   F-statistic:                     2807.
Date:                Fri, 19 Jun 2026   Prob (F-statistic):               0.00
Time:                        13:02:29   Log-Likelihood:                -712.28
No. Observations:                1000   AIC:                             1433.
Df Residuals:                     996   BIC:                             1452.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -0.0258      0.032     -0.808   

In [10]:
# Dataset B
df_b['interaction2'] = df_b['group2'] * df_b['time2']

# Fit the linear regression model with robust standard errors
model_b = smf.ols(
    "outcome2 ~ group2 + time2 + interaction2",
    data=df_b
).fit(cov_type="HC1")

print(model_b.summary())

# differences in differences
mean_b = df_b.groupby(['group2', 'time2'])['outcome2'].mean()
did_b = (mean_b[1][1] - mean_b[1][0]) - (mean_b[0][1] - mean_b[0][0])

print("Differences in Differences for Dataset B")
print(f"Control (0) Before (0): {mean_b[0][0]:.4f}")
print(f"Control (0) After (1):  {mean_b[0][1]:.4f}")
print(f"Treated (1) Before (0): {mean_b[1][0]:.4f}")
print(f"Treated (1) After (1):  {mean_b[1][1]:.4f}")
print(f"Treatment Effect: {did_b:.4f}")

                            OLS Regression Results                            
Dep. Variable:               outcome2   R-squared:                       0.663
Model:                            OLS   Adj. R-squared:                  0.662
Method:                 Least Squares   F-statistic:                     620.2
Date:                Fri, 19 Jun 2026   Prob (F-statistic):          2.62e-227
Time:                        13:02:33   Log-Likelihood:                -1567.5
No. Observations:                1000   AIC:                             3143.
Df Residuals:                     996   BIC:                             3163.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.1021      0.076      1.352   

In [13]:
# We know A had effect 0.6858, and B had effect 1.3499 so B is the largest effect

Q3. Which dataset likely has the largest treatment effect, assuming that the treatment and control groups have parallel trends? 

A3. Dataset 2

Q4. Using the standard errors for regression, which dataset has the most statistically significant (and nonzero) treatment effect? 

In [14]:
dataset_1_coef = model_a.params['interaction1']
dataset_1_standard_error = model_a.bse['interaction1']
dataset_1_coef, dataset_1_standard_error

(np.float64(0.6858469689823519), np.float64(0.06251264175031618))

In [15]:
dataset_2_coef = model_b.params['interaction2']
dataset_2_standard_error = model_b.bse['interaction2']
dataset_2_coef, dataset_2_standard_error

(np.float64(1.3498589246796606), np.float64(0.14703216660376472))

In [16]:
dataset_1_t_stat = dataset_1_coef / dataset_1_standard_error
dataset_2_t_stat = dataset_2_coef / dataset_2_standard_error
dataset_1_t_stat, dataset_2_t_stat

(np.float64(10.971332354209506), np.float64(9.180704847513944))

A. Dataset 1

Q5. Which of these is closest to the treatment effect for group 2? 

A5. It should be around 9.18